# Chapter 16 — Externalize Working Memory

**Companion to Applied AI**

Question: Why is working state in Python variables insufficient — and what replaces it?

By the end of this notebook you will have:

- built a small durable ledger with SQLite
- stopped mid-process and resumed from the record
- shown restart is not resume unless the record rules out a provider effect

## What this notebook demonstrates
A process that survives the death of its own kernel: state lives in SQLite (a file-backed ledger here uses a temp file so crash-simulation is honest).

In [1]:
SEED = 42
import random
random.seed(SEED)
print("seed:", SEED)
import sqlite3, tempfile, os, json

seed: 42


## 1. Durable work state

In [2]:
path = os.path.join(tempfile.gettempdir(), "applied-ai-16-ledger.db")
if os.path.exists(path):
    os.remove(path)
db = sqlite3.connect(path)
db.execute("CREATE TABLE work(task_id TEXT PRIMARY KEY, step TEXT, payload TEXT)")
db.execute("CREATE TABLE receipts(id INTEGER PRIMARY KEY, task_id TEXT, note TEXT)")
db.commit()
db.execute("INSERT INTO work VALUES ('t-9', 'manifest-written', ?)", (json.dumps({"call": "c-1"}),))
db.commit()
print("work state:", db.execute("SELECT * FROM work").fetchall())

work state: [('t-9', 'manifest-written', '{"call": "c-1"}')]


## 2. Kill the process conceptually — then resume from the ledger

In [3]:
db.close()
del db  # the 'kernel' is gone; variables are gone
db2 = sqlite3.connect(path)
step = db2.execute("SELECT step FROM work WHERE task_id='t-9'").fetchone()[0]
print("resumed at step:", step)
assert step == "manifest-written"

resumed at step: manifest-written


## 3. Resume rule: a receipt proves the provider was never touched

In [4]:
n_receipts = db2.execute("SELECT COUNT(*) FROM receipts WHERE task_id='t-9'").fetchone()[0]
if n_receipts == 0:
    print("no provider effect recorded -> safe to continue the call")
    db2.execute("INSERT INTO receipts VALUES (NULL, 't-9', 'continued c-1')")
    db2.commit()
else:
    print("provider effect possible -> must reconcile, never blindly redo")
assert db2.execute("SELECT COUNT(*) FROM receipts WHERE task_id='t-9'").fetchone()[0] == 1
db2.close()
os.remove(path)

no provider effect recorded -> safe to continue the call


## Interpretation
- Supports: restart ≠ resume; resume needs a record proving which effects did or did not happen.
- Does NOT support: a distributed exactly-once claim.

## Try it yourself
1. Insert a receipt *before* the kill and show resume refuses to redo.
2. Store the ledger in memory (`:memory:`) and show there is nothing to resume from.
3. Add a `reconcile` step that queries the (mock) provider before continuing.